# 04 · Feature Engineering — *from scratch*

**Phase goal:** turn the raw columns into a compact, numeric feature matrix that
*every* model family can consume — without leaking the target.

## The core decision: **target-encode** `make`/`model`, don't one-hot them
`model` has thousands of levels. One-hot encoding would create thousands of
columns — huge, slow, and unusable by some models. **Target encoding** replaces
each category with a number: *the average price of cars in that category*. That
turns two text columns into two dense numeric columns.

The catch is **leakage**: if we compute a category's mean using a row's *own*
price, the model peeks at the answer. The fix is **cross-fitting** — compute each
row's encoding from the *other* rows only. scikit-learn's `TargetEncoder` does
this for us; we'll first build the intuition by hand.

In [1]:
import warnings
warnings.filterwarnings("ignore")   # hide library deprecation notices for clean output

import numpy as np
import pandas as pd

DATA_PATH = "../data/raw/cars24-car-price-cleaned-new.csv.gz"

# --- The feature schema, written out (normally in config.py) ---------------
TARGET        = "selling_price"
TARGET_ENCODE = ["make", "model"]                                   # -> target-encoded
NUMERIC       = ["km_driven", "mileage", "engine", "max_power", "age"]
FLAGS         = ["Individual", "Trustmark Dealer",                  # already 0/1
                 "Diesel", "Electric", "LPG", "Petrol",
                 "Manual", "Seats_5", "Seats_Above_5"]
FEATURES      = TARGET_ENCODE + NUMERIC + FLAGS                     # final column order


def clean(df):
    df = df.rename(columns={"5": "Seats_5", ">5": "Seats_Above_5"})
    df = df.dropna(subset=[TARGET])
    df = df[df[TARGET] > 0].dropna()
    for col in TARGET_ENCODE:
        df[col] = df[col].astype(str).str.strip().str.upper()
    return df.reset_index(drop=True)


df = clean(pd.read_csv(DATA_PATH))
print(f"{len(df):,} rows | {len(FEATURES)} model features")

19,820 rows | 16 model features


### Build the intuition: naive mean-encoding (the leaky version)
Just to *see* the idea, here is the simplest possible target encoding of
`make` — replace each brand with the mean price of that brand. **Don't use this
for training** (it's leaky); it's only here to make the concept concrete.

In [2]:
# Average price per brand -> a single number that stands in for the text.
make_means = df.groupby("make")[TARGET].mean()
demo = df[["make", TARGET]].head(5).copy()
demo["make_encoded"] = demo["make"].map(make_means).round(2)
demo

,make,selling_price,make_encoded
0,MARUTI,1.20,4.68
1,HYUNDAI,5.50,5.46
2,HYUNDAI,2.15,5.46
3,MARUTI,2.26,4.68
4,FORD,5.70,5.86


### Why one-hot would be a bad idea here
Count how many columns each approach produces.

In [3]:
onehot_cols = df["make"].nunique() + df["model"].nunique()
print(f"One-hot make+model would add ~{onehot_cols:,} columns")
print(f"Target encoding adds exactly {len(TARGET_ENCODE)} columns (make, model)")
print(f"Total model features either way besides those: {len(NUMERIC) + len(FLAGS)}")

One-hot make+model would add ~3,274 columns
Target encoding adds exactly 2 columns (make, model)
Total model features either way besides those: 14


### The real, leakage-safe preprocessor
We wrap everything in a scikit-learn **`ColumnTransformer`**:

- `make`, `model` → **`TargetEncoder`** (cross-fitted internally → no leakage),
- the numeric and flag columns → **passed through unchanged**.

Building it as one object means the *exact* same transformation is applied at
training time and at serving time — no chance of them drifting apart.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

RANDOM_STATE = 42


def build_preprocessor():
    """Target-encode make/model; pass numeric + flag columns through as-is."""
    return ColumnTransformer(
        transformers=[
            ("target_enc",
             TargetEncoder(target_type="continuous", random_state=RANDOM_STATE),
             TARGET_ENCODE),
            ("passthrough", "passthrough", NUMERIC + FLAGS),
        ],
        remainder="drop",
    )


# X = the feature columns (in a fixed order); y = the target price.
X = df[FEATURES].copy()
y = df[TARGET].copy()

pre = build_preprocessor()
X_encoded = pre.fit_transform(X, y)      # fit_transform because we pass y in

print("encoded matrix shape:", X_encoded.shape, "  (all numeric now)")
pd.DataFrame(X_encoded[:3], columns=FEATURES).round(2)

encoded matrix shape: (19820, 16)   (all numeric now)


,make,model,km_driven,mileage,engine,max_power,age,Individual,Trustmark Dealer,Diesel,Electric,LPG,Petrol,Manual,Seats_5,Seats_Above_5
0,4.70,1.14,120000.0,19.7,796.0,46.3,11.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,5.43,4.73,20000.0,18.9,1197.0,82.0,7.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
2,5.44,3.50,60000.0,17.0,1197.0,80.0,13.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0


### The price bands, from scratch
The Low / Medium / High band is **derived from the price**, not predicted
separately. We split the price range into **terciles** (three equal-count
buckets) and label them. Both functions are written out below.

In [5]:
BAND_LABELS = ["Low", "Medium", "High"]


def band_edges(prices):
    """Return the tercile cut points [min, 33%, 67%, max] of the prices."""
    quantiles = [0.0, 1/3, 2/3, 1.0]
    edges = [float(np.quantile(prices, q)) for q in quantiles]
    # Guard against ties so the edges are strictly increasing.
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 0.01
    return [round(e, 2) for e in edges]


def price_to_band(prices, edges):
    """Map price(s) to 'Low'/'Medium'/'High' using the two inner tercile edges."""
    prices = np.asarray(prices, dtype=float)
    # np.digitize with the inner edges gives an index 0/1/2 -> a label.
    idx = np.clip(np.digitize(prices, edges[1:-1]), 0, len(BAND_LABELS) - 1)
    return np.array(BAND_LABELS)[idx]


edges = band_edges(y)
print("band edges (Lakhs):", edges)
print("example prices [2, 5, 25] ->", list(price_to_band([2.0, 5.0, 25.0], edges)))

band edges (Lakhs): [0.3, 3.99, 6.75, 20.9]
example prices [2, 5, 25] -> ['Low', 'Medium', 'High']


### Why this is production-grade
- **Compact:** ~16 numeric features instead of thousands → a small, fast model.
- **Leakage-safe:** `TargetEncoder` cross-fits internally; in cross-validation
  (nb 05) we also fit it *inside each fold*.
- **Universal:** the dense numeric output works for every model, including
  gradient boosting.
- **One contract:** the encoder lives *inside* one pipeline object, so train-time
  and serve-time preprocessing are guaranteed identical.